# KuchoLM NIDA-7M — fast copy curriculum
反復崩壊とコピー失敗を抑える実験版。60k件、COPY warmup 1 pass + mixed 2 pass、EOS重み付け、no-repeat 3-gramを使用します。


In [ ]:
!pip -q install mecab-python3 unidic-lite datasets sentencepiece torch


In [ ]:
from pathlib import Path
import difflib,json,math,random,re,string
import MeCab,sentencepiece as spm,torch
from datasets import load_dataset
from torch import nn
from torch.utils.data import Dataset,DataLoader
DATA_PATH=Path('/content/kucholm_nida_fast.jsonl'); WORK_DIR=Path('/content/kucholm_work_fast'); WORK_DIR.mkdir(exist_ok=True)
MAX_ROWS=60_000; VOCAB_SIZE=12_000; MAX_LEN=160; COPY_MIX_RATIO=.50; MIXED_PASSES=2; EOS_WEIGHT=3.0; SEED=42
random.seed(SEED); torch.manual_seed(SEED); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); tagger=MeCab.Tagger()
print('device:',device)


In [ ]:
URL_RE=re.compile(r'https?://|www\.|```|`[^`]+`'); SENTENCE_RE=re.compile(r'(.+?[。！？!?]+|.+$)',re.S)
def parse_tokens(text):
    node=tagger.parseToNode(text); out=[]
    while node:
        if node.surface:
            f=node.feature.split(','); out.append((node.surface,f[0] if len(f)>0 else '',f[4] if len(f)>4 else '*',f[7] if len(f)>7 else '*',f[10] if len(f)>10 else '*'))
        node=node.next
    return out
def dictionary_form(t):
    for v in (t[4],t[3]):
        if v not in ('','*') and re.search(r'[ぁ-ん一-龯]',v): return v
    return t[0]
def ta_form(base,t):
    if base=='行く': return '行った'
    if base=='来る': return '来た'
    if base=='する': return 'した'
    if '一段' in t[2]: return base[:-1]+'た'
    if base.endswith(('う','つ','る')): return base[:-1]+'った'
    if base.endswith(('む','ぶ','ぬ')): return base[:-1]+'んだ'
    if base.endswith('く'): return base[:-1]+'いた'
    if base.endswith('ぐ'): return base[:-1]+'いだ'
    if base.endswith('す'): return base[:-1]+'した'
    return base+'た'
def convert_tail(body):
    for p,r in [(r'かもしれません$','かもしれない'),(r'わかりません$','わからない'),(r'知りません$','知らない'),(r'ありません$','ない'),(r'ございました$','あった'),(r'ございます$','ある'),(r'てきました$','てきた'),(r'て来ました$','て来た'),(r'てしまいました$','てしまった'),(r'でしまいました$','でしまった')]:
        if re.search(p,body): return re.sub(p,r,body)
    ts=parse_tokens(body); surfaces=[t[0] for t in ts]
    for suffix in (['まし','た'],['ます']):
        if len(surfaces)>=len(suffix) and surfaces[-len(suffix):]==suffix:
            end=len(ts)-len(suffix); vi=next((i for i in range(end-1,-1,-1) if ts[i][1]=='動詞'),None)
            if vi is not None:
                base=dictionary_form(ts[vi]); prefix=''.join(t[0] for t in ts[:vi]); return prefix+(ta_form(base,ts[vi]) if len(suffix)==2 else base)
    if body.endswith('でした'): return body[:-3]+'だった'
    if body.endswith('です'): return body[:-2]
    return body
def convert_sentence(s):
    m=re.match(r'^(\s*)(.*?)(\s*)$',s,re.S); leading,core,trailing=m.groups()
    if not core or URL_RE.search(core): return s
    pm=re.search(r'([。！？!?]+)$',core); punct=pm.group(1) if pm else ''; body=core[:-len(punct)] if punct else core; q=bool(re.search(r'[？?]$',punct))
    body=convert_tail(body); return leading+body+('ニカ' if q else 'ニダよ')+punct+trailing
def to_nida(text):
    if not text or URL_RE.search(text): return None
    return ''.join(convert_sentence(m.group(0)) for m in SENTENCE_RE.finditer(text))
if not DATA_PATH.exists():
    ds=load_dataset('range3/cc100-ja',split='train',streaming=True); n=0
    with DATA_PATH.open('w',encoding='utf-8') as f:
        for row in ds:
            src=str(row['text'])
            if len(src.strip())<2 or len(src)>220: continue
            tgt=to_nida(src)
            if not tgt or tgt==src: continue
            f.write(json.dumps({'source':src,'target':tgt},ensure_ascii=False)+'\n'); n+=1
            if n>=MAX_ROWS: break
    print('written:',n)
else: print('using:',DATA_PATH)


In [ ]:
rows=[]
with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        x=json.loads(line); rows.append((x['source'],x['target']))
random.shuffle(rows); cut=max(1,int(len(rows)*.98)); train_raw=rows[:cut]; val_raw=rows[cut:]
copy_rows=[(f'<COPY> {s}',s) for s,_ in train_raw]; mixed=[]
for s,t in train_raw:
    mixed.append((f'<NIDA_FICTION> {s}',t))
    if random.random()<COPY_MIX_RATIO: mixed.append((f'<COPY> {s}',s))
val_rows=[(f'<NIDA_FICTION> {s}',t) for s,t in val_raw]
spm_input=WORK_DIR/'spm.txt'
with spm_input.open('w',encoding='utf-8') as f:
    for a,b in mixed:
        f.write(a.replace('\n',' ')+'\n'+b.replace('\n',' ')+'\n')
spm.SentencePieceTrainer.train(input=str(spm_input),model_prefix=str(WORK_DIR/'spm'),vocab_size=VOCAB_SIZE,model_type='bpe',character_coverage=1.0,byte_fallback=True,normalization_rule_name='identity',split_digits=True,hard_vocab_limit=False,pad_id=0,unk_id=1,bos_id=2,eos_id=3,user_defined_symbols=['<NIDA_FICTION>','<COPY>'])
sp=spm.SentencePieceProcessor(model_file=str(WORK_DIR/'spm.model')); PAD,UNK,BOS,EOS=0,1,2,3; VOCAB=sp.vocab_size(); print('vocab:',VOCAB)
def enc(x): return [BOS]+sp.encode(x,out_type=int)+[EOS]
def fits(p): return len(enc(p[0]))<=MAX_LEN and len(enc(p[1]))<=MAX_LEN
copy_rows=[p for p in copy_rows if fits(p)]; mixed=[p for p in mixed if fits(p)]; val_rows=[p for p in val_rows if fits(p)]
class Pairs(Dataset):
    def __init__(self,x): self.x=x
    def __len__(self): return len(self.x)
    def __getitem__(self,i): a,b=self.x[i]; return torch.tensor(enc(a)),torch.tensor(enc(b))
def collate(batch):
    a,b=zip(*batch); return nn.utils.rnn.pad_sequence(a,batch_first=True,padding_value=PAD),nn.utils.rnn.pad_sequence(b,batch_first=True,padding_value=PAD)
BATCH=96 if device.type=='cuda' else 8; kw=dict(batch_size=BATCH,collate_fn=collate,pin_memory=device.type=='cuda',num_workers=2 if device.type=='cuda' else 0)
copy_loader=DataLoader(Pairs(copy_rows),shuffle=True,**kw); mixed_loader=DataLoader(Pairs(mixed),shuffle=True,**kw); val_loader=DataLoader(Pairs(val_rows),shuffle=False,**kw)
print('rows:',len(copy_rows),len(mixed),len(val_rows),'batch:',BATCH)


In [ ]:
D_MODEL=224; NHEAD=8; ENC_LAYERS=3; DEC_LAYERS=3; FF=896; LR=3e-4
class KuchoTransformer(nn.Module):
    def __init__(self):
        super().__init__(); self.embed=nn.Embedding(VOCAB,D_MODEL,padding_idx=PAD); self.pos=nn.Embedding(MAX_LEN,D_MODEL)
        self.tf=nn.Transformer(d_model=D_MODEL,nhead=NHEAD,num_encoder_layers=ENC_LAYERS,num_decoder_layers=DEC_LAYERS,dim_feedforward=FF,dropout=.1,batch_first=True,norm_first=True)
        self.lm_head=nn.Linear(D_MODEL,VOCAB,bias=False); self.lm_head.weight=self.embed.weight
    def add_pos(self,x):
        p=torch.arange(x.size(1),device=x.device).unsqueeze(0); return self.embed(x)*math.sqrt(D_MODEL)+self.pos(p)
    def forward(self,src,tgt):
        mask=nn.Transformer.generate_square_subsequent_mask(tgt.size(1),device=tgt.device)
        h=self.tf(self.add_pos(src),self.add_pos(tgt),tgt_mask=mask,src_key_padding_mask=src.eq(PAD),tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=src.eq(PAD)); return self.lm_head(h)
model=KuchoTransformer().to(device); print(f'{sum(p.numel() for p in model.parameters())/1e6:.3f}M parameters')
@torch.no_grad()
def infer(text,tag='<NIDA_FICTION>'):
    src_ids=enc(f'{tag} {text}'); src=torch.tensor([src_ids],device=device); out=[BOS]; limit=min(MAX_LEN-1,max(12,len(src_ids)+12))
    for _ in range(limit):
        logits=model(src,torch.tensor([out],device=device))[0,-1].clone()
        if len(out)>=3:
            prefix=tuple(out[-2:]); banned={out[i+2] for i in range(len(out)-2) if tuple(out[i:i+2])==prefix}
            if banned: logits[list(banned)]=-float('inf')
        if len(out)>=2 and out[-1]==out[-2]: logits[out[-1]]=-float('inf')
        nxt=int(torch.argmax(logits))
        if nxt==EOS: break
        out.append(nxt)
    return sp.decode(out[1:])
TESTS=['今日は学校です。','明日は雨が降るかもしれません。','最近少し暖かくなってきました。','製品KuchoLM-X7-2026は正常に動作しています。','髙﨑𠮷野家ABC-123を確認しました。']


In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,betas=(.9,.98),weight_decay=.01); scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda')
def loss_fn(logits,target):
    y=target.reshape(-1); l=nn.functional.cross_entropy(logits.reshape(-1,VOCAB),y,ignore_index=PAD,reduction='none'); valid=y!=PAD; w=torch.ones_like(l); w[y==EOS]=EOS_WEIGHT; return (l[valid]*w[valid]).sum()/w[valid].sum()
def train_pass(loader):
    model.train(); total=0
    for src,tgt in loader:
        src,tgt=src.to(device,non_blocking=True),tgt.to(device,non_blocking=True); optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=device.type=='cuda'): logits=model(src,tgt[:,:-1]); loss=loss_fn(logits,tgt[:,1:])
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update(); total+=loss.item()
    return total/max(1,len(loader))
@torch.no_grad()
def val_loss():
    model.eval(); total=0
    for src,tgt in val_loader:
        src,tgt=src.to(device,non_blocking=True),tgt.to(device,non_blocking=True); total+=loss_fn(model(src,tgt[:,:-1]),tgt[:,1:]).item()
    return total/max(1,len(val_loader))
def show(label):
    model.eval(); print('---',label,'---')
    for s in TESTS: print(s,'->',infer(s))
print('copy warmup loss:',train_pass(copy_loader)); torch.save({'model':model.state_dict()},WORK_DIR/'copy-warmup.pt'); show('COPY warmup')
best=float('inf'); best_path=WORK_DIR/'KuchoLM-NIDA-7M.pt'
for epoch in range(1,MIXED_PASSES+1):
    tr=train_pass(mixed_loader); va=val_loss(); torch.save({'model':model.state_dict(),'val':va},WORK_DIR/f'epoch{epoch}.pt'); print(f'epoch {epoch}: train={tr:.4f} val={va:.4f}'); show(f'epoch {epoch}')
    if va<best: best=va; torch.save({'model':model.state_dict(),'best_val':best},best_path)
model.load_state_dict(torch.load(best_path,map_location=device)['model']); show('BEST')
def sim(a,b): return difflib.SequenceMatcher(None,a,b).ratio()
scores=[]
for tagged,expected in val_rows[:100]: scores.append(sim(expected,infer(tagged.removeprefix('<NIDA_FICTION> '))))
print('char similarity:',sum(scores)/max(1,len(scores))); print('model:',best_path); print('tokenizer:',WORK_DIR/'spm.model')
